# MobileADAS3D-S1 GT-only baseline

This notebook trains the deployable MobileNetV4 S1 student using KITTI ground truth only. Knowledge distillation is disabled. Run the 20-epoch health gate first; continuation to epoch 100 is deliberately locked until the gate metrics are reviewed. Select a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import json, os, shlex, shutil, subprocess, sys
REPO_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/mobile_adas3d')
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
DATASET_VIEW = Path('/content/kitti_s1')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/mobileadas3d_s1_gt_baseline')
R0_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
BASE_CONFIG = PROJECT_DIR/'configs/kitti_mobileadas3d_s1_gt_baseline.yaml'
RUNTIME_CONFIG_DIR = PROJECT_DIR/'configs/runtime_s1_gt'
RUN_NAME = 'mobileadas3d_s1_gt_baseline'
GATE_EPOCHS = 20
FULL_EPOCHS = 100
def run(command, cwd=None):
    command = [str(x) for x in command]; print('+', shlex.join(command), flush=True)
    result = subprocess.run(command, cwd=cwd)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')

In [ ]:
# Repository, dependencies, and GPU. The required local commits must be pushed before this clone.
if not (PROJECT_DIR/'.git').exists(): run(['git','clone','--branch',BRANCH,REPO_URL,PROJECT_DIR])
else:
    run(['git','fetch','origin'], cwd=PROJECT_DIR)
    run(['git','checkout',BRANCH], cwd=PROJECT_DIR)
    run(['git','pull','--ff-only','origin',BRANCH], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'], cwd=PROJECT_DIR)
import torch, timm
if not torch.cuda.is_available(): raise RuntimeError('Choose Runtime > Change runtime type > GPU')
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=PROJECT_DIR, text=True).strip())
print('torch/timm:', torch.__version__, timm.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Prefer an already-complete local stage; otherwise create a zero-copy canonical Drive view.
def count_files(path, suffix): return sum(1 for p in path.iterdir() if p.is_file() and p.suffix == suffix) if path.is_dir() else 0
def complete(root):
    return count_files(root/'training/image_2','.png') == 7481 and count_files(root/'training/label_2','.txt') == 7481 and count_files(root/'training/calib','.txt') == 7481
if complete(LOCAL_DATASET_ROOT):
    DATASET_ROOT = LOCAL_DATASET_ROOT
    print('Using complete local KITTI stage:', DATASET_ROOT)
else:
    aliases = {'image_2':['image_2','image_02'], 'label_2':['label_2','label_02'], 'calib':['calib']}
    (DATASET_VIEW/'training').mkdir(parents=True, exist_ok=True)
    for canonical, candidates in aliases.items():
        source = next((DRIVE_DATASET_ROOT/'training'/name for name in candidates if (DRIVE_DATASET_ROOT/'training'/name).is_dir()), None)
        if source is None: raise FileNotFoundError(f'Missing source for {canonical}: {candidates}')
        link = DATASET_VIEW/'training'/canonical
        if link.is_symlink() and link.resolve() == source.resolve(): continue
        if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace unexpected {link}')
        link.symlink_to(source, target_is_directory=True)
    DATASET_ROOT = DATASET_VIEW
    if not complete(DATASET_ROOT): raise RuntimeError('Canonical KITTI Drive view is incomplete')
    print('Using zero-copy Drive KITTI view:', DATASET_ROOT)
print('Counts:', {name: count_files(DATASET_ROOT/'training'/name, suffix) for name,suffix in [('image_2','.png'),('label_2','.txt'),('calib','.txt')]})

In [ ]:
# Fail-closed R0 provenance, GT-only configs, split/taxonomy audit, and one real CUDA loss.
if not R0_SELECTION.is_file(): raise FileNotFoundError(R0_SELECTION)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run([sys.executable,'scripts/prepare_s1_gt_baseline.py','--base-config',BASE_CONFIG,'--r0-selection',R0_SELECTION,'--output-dir',OUTPUT_DIR,'--config-dir',RUNTIME_CONFIG_DIR,'--run-name',RUN_NAME,'--gate-epochs',GATE_EPOCHS,'--full-epochs',FULL_EPOCHS], cwd=PROJECT_DIR)
GATE_CONFIG = RUNTIME_CONFIG_DIR/'mobileadas3d_s1_gt_gate20.yaml'
FULL_CONFIG = RUNTIME_CONFIG_DIR/'mobileadas3d_s1_gt_full100.yaml'
COMMON = ['--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--output-dir',OUTPUT_DIR]
run([sys.executable,'scripts/check_training_ready.py','--config',GATE_CONFIG,*COMMON,'--require-cuda','--report',OUTPUT_DIR/'training_preflight.json'], cwd=PROJECT_DIR)
manifest = json.loads((OUTPUT_DIR/'s1_gt_baseline_manifest.json').read_text())
assert manifest['distillation_enabled'] is False and manifest['architecture'] == 'MobileADAS3D-S1'
print(json.dumps(manifest, indent=2))

## Twenty-epoch health gate

This is real S1 training. It saves `latest.pt` every epoch and epoch checkpoints every five epochs. Re-running the cell resumes the newest matching run.

In [ ]:
from collections import deque
def run_streamed(command, cwd, log_path):
    command=[str(x) for x in command]; log_path.parent.mkdir(parents=True,exist_ok=True)
    print('+',shlex.join(command)); print('Durable log:',log_path)
    tail=deque(maxlen=120); env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=p.wait()
    if code: raise RuntimeError(f'Exit {code}; log={log_path}\n'+'\n'.join(tail))
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
RESUME=candidates[-1] if candidates else None
if RESUME:
    payload=torch.load(RESUME,map_location='cpu',weights_only=False)
    if payload.get('epoch',0) >= GATE_EPOCHS: print(f'Gate already complete at epoch {payload["epoch"]}: {RESUME}')
    else:
        run_streamed([sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',GATE_CONFIG,*COMMON,'--run-name',RUN_NAME,'--resume',RESUME],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'gate_resume_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
else:
    run_streamed([sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',GATE_CONFIG,*COMMON,'--run-name',RUN_NAME],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'gate_fresh_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
LATEST_CHECKPOINT=candidates[-1]; TRAIN_RUN_DIR=LATEST_CHECKPOINT.parent.parent
print('Gate checkpoint:',LATEST_CHECKPOINT); print('Run directory:',TRAIN_RUN_DIR)

In [ ]:
# Full product-taxonomy evaluation of the epoch-20 gate.
GATE_EVAL_DIR=TRAIN_RUN_DIR/'kitti_r40_gate20'
run([sys.executable,'scripts/evaluate_kitti_r40.py','--config',GATE_CONFIG,*COMMON,'--checkpoint',LATEST_CHECKPOINT,'--split','val','--score-threshold','0.001','--topk','300','--nms-iou-threshold','0.5','--output-dir',GATE_EVAL_DIR], cwd=PROJECT_DIR)
import pandas as pd
gate_summary=json.loads((GATE_EVAL_DIR/'kitti_r40_summary.json').read_text())
assert gate_summary['complete_split'] and gate_summary['evaluated_images']==3769
gate_metrics=pd.DataFrame(gate_summary['metrics'])
display(gate_metrics.pivot_table(index=['metric','class_name'],columns='difficulty',values='ap_r40').round(3))
print('Send this table and the final training summary for review before continuation.')

## Controlled continuation to epoch 100

Do not run until the epoch-20 loss and product AP health gate is reviewed. Set the authorization flag to `True` only after that decision. Resume preserves model, optimizer, scaler, scheduler, and global step.

In [ ]:
AUTHORIZE_CONTINUATION = False
if not AUTHORIZE_CONTINUATION: raise RuntimeError('Review epoch-20 gate before authorizing continuation')
payload=torch.load(LATEST_CHECKPOINT,map_location='cpu',weights_only=False)
if payload['epoch'] < GATE_EPOCHS: raise RuntimeError(f'Gate incomplete: epoch {payload["epoch"]}')
if payload['epoch'] >= FULL_EPOCHS: print('Full run already complete')
else:
    run_streamed([sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',FULL_CONFIG,*COMMON,'--run-name',RUN_NAME,'--resume',LATEST_CHECKPOINT],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'full_resume_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')